In [2]:
# ============================================================
# EPL Transformer Trajectory Model (NO-ODDS) — RESUMABLE (IMPROVED + DATASET-ROBUST)
# JUPYTER CELL VERSION (NO argparse) — includes:
# ✅ Robust column mapping
# ✅ Strictly-causal rolling form features (if present)
# ✅ Strictly-causal league table context
# ✅ Transformer w/ collision features
# ✅ Early stopping on TEST log-loss + best checkpoint
# ✅ Temperature scaling calibration
# ✅ Fixture prediction with TRUE "as-of date" context + TRUE "as-of date" rolling form
#
# Drop this entire cell into Jupyter and run.
# ============================================================

import os, math, random, glob, re
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -------------------- CONFIG --------------------
CSV_PATH    = "epl_all_seasons_football_data.csv"
MIN_DATE    = "2005-01-01"
NAN_THRESH  = 0.90
RANDOM_SEED = 42

SEQ_LEN     = 8
BATCH_SIZE  = 256
EPOCHS      = 40
LR          = 2e-4
WEIGHT_DECAY= 1e-2

D_MODEL     = 128
N_HEAD      = 4
N_LAYERS    = 4
D_FF        = 512
DROPOUT     = 0.3

LABEL_SMOOTHING = 0.05
EARLY_STOP_PATIENCE = 6
MIN_EPOCHS_BEFORE_STOP = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RUN_NAME = "epl_traj_v3"
CKPT_ROOT = "checkpoints_epl_traj"
CKPT_PREFIX = "epl_traj"
CKPT_DIR = os.path.join(CKPT_ROOT, RUN_NAME)
os.makedirs(CKPT_DIR, exist_ok=True)

RESUME = True
SAVE_EVERY_N = 10
SAVE_KEEP_LAST_K = 3
SAVE_BEST_EVERY_N = 10

# ---- fixtures to predict ----
fixtures = [
    ("2026-02-28", "Wolves",       "Aston Villa"),
    ("2026-02-28", "Bournemouth",  "Sunderland"),
    ("2026-02-28", "Newcastle",    "Everton"),
    ("2026-02-28", "Burnley",      "Brentford"),
    ("2026-02-28", "Liverpool",    "West Ham"),
    ("2026-02-28", "Leeds",        "Man City"),
]
fixtures_df = pd.DataFrame(fixtures, columns=["date","home_team","away_team"])
fixtures_df["date"] = pd.to_datetime(fixtures_df["date"])

# -------------------- helper: team name cleaning --------------------
def clean_team_name(x: str) -> str:
    if pd.isna(x):
        return x
    s = str(x).strip()
    repl = {
        "Wolverhampton": "Wolves",
        "Manchester City": "Man City",
        "Manchester United": "Man United",
        "West Ham United": "West Ham",
        "Nott'm Forest": "Nottm Forest",
        "Nottingham Forest": "Nottm Forest",
        "Spurs": "Tottenham",
        "Tottenham Hotspur": "Tottenham",
    }
    return repl.get(s, s)

def result_points(hg, ag):
    if hg > ag:  return 3, 0
    if hg < ag:  return 0, 3
    return 1, 1

# -------------------- load + normalize columns --------------------
df0 = pd.read_csv(CSV_PATH, low_memory=False)

# date
if "date" not in df0.columns:
    for alt in ["Date", "match_date", "MatchDate"]:
        if alt in df0.columns:
            df0 = df0.rename(columns={alt: "date"})
            break
df0["date"] = pd.to_datetime(df0["date"], errors="coerce")

# team / goals / result
rename_map = {
    "HomeTeam": "home_team", "AwayTeam": "away_team",
    "FTHG": "home_goals", "FTAG": "away_goals",
    "FTR": "ft_result",
    "Home": "home_team", "Away": "away_team",
}
for k, v in rename_map.items():
    if k in df0.columns and v not in df0.columns:
        df0 = df0.rename(columns={k: v})

# drop very-missing columns
df0 = df0.loc[:, df0.isna().mean() <= NAN_THRESH].copy()

need_base = ["date", "home_team", "away_team", "home_goals", "away_goals"]
missing = [c for c in need_base if c not in df0.columns]
if missing:
    raise ValueError(f"Missing required base columns: {missing}\nColumns start: {list(df0.columns)[:40]}")

# clean team names
df0["home_team"] = df0["home_team"].map(clean_team_name)
df0["away_team"] = df0["away_team"].map(clean_team_name)

# goals numeric
df0["home_goals"] = pd.to_numeric(df0["home_goals"], errors="coerce")
df0["away_goals"] = pd.to_numeric(df0["away_goals"], errors="coerce")
df0 = df0[df0["home_goals"].notna() & df0["away_goals"].notna()].copy()

# ft_result + target
if "ft_result" not in df0.columns:
    hg = df0["home_goals"].astype(int)
    ag = df0["away_goals"].astype(int)
    df0["ft_result"] = np.where(hg > ag, "H", np.where(hg < ag, "A", "D"))
else:
    df0["ft_result"] = df0["ft_result"].astype(str).str.strip().str.upper()
    df0 = df0[df0["ft_result"].isin(["H","D","A"])].copy()

if "target" not in df0.columns:
    df0["target"] = df0["ft_result"].astype(str).str.strip().str.upper()
else:
    df0["target"] = df0["target"].astype(str).str.strip().str.upper()
    df0 = df0[df0["target"].isin(["H","D","A"])].copy()

# filter by date
df0 = df0[df0["date"].notna() & (df0["date"] >= pd.to_datetime(MIN_DATE))].copy()
df0 = df0.sort_values("date").reset_index(drop=True)

print(f"Loaded: {CSV_PATH}")
print("Rows after cleanup:", len(df0))
print("Date range:", df0["date"].min().date(), "->", df0["date"].max().date())

# ============================================================
# 1) Build long per-team history (base trajectory tokens)
# ============================================================
rows = []
for i, r in df0.iterrows():
    hg, ag = float(r["home_goals"]), float(r["away_goals"])
    hp, ap = result_points(hg, ag)
    dt = r["date"]

    rows.append({
        "match_idx": i, "date": dt, "team": r["home_team"], "opp": r["away_team"],
        "is_home": 1.0, "gf": hg, "ga": ag, "gd": hg - ag, "pts": float(hp),
    })
    rows.append({
        "match_idx": i, "date": dt, "team": r["away_team"], "opp": r["home_team"],
        "is_home": 0.0, "gf": ag, "ga": hg, "gd": ag - hg, "pts": float(ap),
    })

long = pd.DataFrame(rows).sort_values(["date","match_idx","team"]).reset_index(drop=True)

# ============================================================
# 1b) Strictly-causal rolling "form" features from match stats (if present)
# ============================================================
STAT_PAIRS = {
    "shots": ("HS","AS"),
    "shots_on_target": ("HST","AST"),
    "corners": ("HC","AC"),
    "yellows": ("HY","AY"),
    "reds": ("HR","AR"),
}
available_stats = {k:v for k,v in STAT_PAIRS.items() if (v[0] in df0.columns and v[1] in df0.columns)}
print("Available stats for rolling form:", list(available_stats.keys()) if available_stats else "(none)")

def coerce_num(s):
    return pd.to_numeric(s, errors="coerce")

form_rows = []
if available_stats:
    tmp = df0.copy()
    for name, (hcol, acol) in available_stats.items():
        tmp[hcol] = coerce_num(tmp[hcol])
        tmp[acol] = coerce_num(tmp[acol])

    for i, r in tmp.iterrows():
        dt = r["date"]
        ht, at = r["home_team"], r["away_team"]

        rec_h = {"match_idx": i, "date": dt, "team": ht}
        rec_a = {"match_idx": i, "date": dt, "team": at}
        for name, (hcol, acol) in available_stats.items():
            hv = r[hcol]; av = r[acol]
            rec_h[f"{name}_for"] = hv
            rec_h[f"{name}_against"] = av
            rec_a[f"{name}_for"] = av
            rec_a[f"{name}_against"] = hv

        form_rows.append(rec_h)
        form_rows.append(rec_a)

    form_long = pd.DataFrame(form_rows).sort_values(["team","date","match_idx"]).reset_index(drop=True)

    ROLL_WINS = [3, 6, 10]
    for name in available_stats.keys():
        for w in ROLL_WINS:
            for side in ["for", "against"]:
                col = f"{name}_{side}"
                out = f"{col}_roll{w}"
                form_long[out] = (
                    form_long.groupby("team")[col]
                    .apply(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
                    .reset_index(level=0, drop=True)
                )

    for name in available_stats.keys():
        for w in ROLL_WINS:
            form_long[f"{name}_diff_roll{w}"] = form_long[f"{name}_for_roll{w}"] - form_long[f"{name}_against_roll{w}"]
else:
    form_long = None

# ============================================================
# 2) Compute league table context strictly causal
# ============================================================
teams = sorted(pd.unique(pd.concat([df0["home_team"], df0["away_team"]], ignore_index=True)))
team_to_id = {t:i for i,t in enumerate(teams)}

points = {t: 0.0 for t in teams}
gf_tot = {t: 0.0 for t in teams}
ga_tot = {t: 0.0 for t in teams}
last_date = {t: None for t in teams}

ctx_rows = []
for i, r in df0.iterrows():
    dt = r["date"]
    ht = r["home_team"]; at = r["away_team"]
    hg = float(r["home_goals"]); ag = float(r["away_goals"])
    hp, ap = result_points(hg, ag)

    table = []
    for t in teams:
        gd = gf_tot[t] - ga_tot[t]
        table.append((t, points[t], gd, gf_tot[t]))
    table_sorted = sorted(table, key=lambda x: (x[1], x[2], x[3]), reverse=True)
    rank_map = {t: (rk+1) for rk, (t,_,_,_) in enumerate(table_sorted)}

    def rest_days(team):
        if last_date[team] is None:
            return np.nan
        return float((dt - last_date[team]).days)

    ctx_rows.append({
        "match_idx": i,
        "home_pts_pre": points[ht],
        "away_pts_pre": points[at],
        "home_rank_pre": float(rank_map[ht]),
        "away_rank_pre": float(rank_map[at]),
        "home_rest_days": rest_days(ht),
        "away_rest_days": rest_days(at),
    })

    points[ht] += hp; points[at] += ap
    gf_tot[ht] += hg; ga_tot[ht] += ag
    gf_tot[at] += ag; ga_tot[at] += hg
    last_date[ht] = dt; last_date[at] = dt

ctx = pd.DataFrame(ctx_rows).set_index("match_idx")
df = df0.join(ctx, how="left")

# ============================================================
# 3) Build sequences + final dataset rows
# ============================================================
long_by_team = {t: long[long["team"] == t].sort_values(["date","match_idx"]).reset_index(drop=True) for t in teams}
TOKEN_COLS = ["is_home", "gf", "ga", "gd", "pts"]

def get_team_seq(team: str, match_idx: int):
    hist = long_by_team[team]
    past = hist[hist["match_idx"] < match_idx]
    if len(past) < SEQ_LEN:
        return None
    return past.iloc[-SEQ_LEN:][TOKEN_COLS].to_numpy(dtype=np.float32)

def build_form_lookup(form_long: pd.DataFrame):
    feat_cols = [c for c in form_long.columns if c not in ["match_idx","date","team"]]
    lookup = {}
    for _, r in form_long.iterrows():
        lookup[(r["team"], int(r["match_idx"]))] = r[feat_cols].to_dict()
    return lookup, feat_cols

if form_long is not None:
    form_lookup, FORM_COLS = build_form_lookup(form_long)
else:
    form_lookup, FORM_COLS = {}, []

def safe_get_form(team: str, match_idx: int):
    d = form_lookup.get((team, int(match_idx)), None)
    if d is None:
        return {c: np.nan for c in FORM_COLS}
    return d

def build_dataset_rows():
    out_rows = []
    for i, r in df.iterrows():
        ht, at = r["home_team"], r["away_team"]
        home_seq = get_team_seq(ht, i)
        away_seq = get_team_seq(at, i)
        if home_seq is None or away_seq is None:
            continue

        rec = {
            "match_idx": i,
            "date": r["date"],
            "home_team": ht,
            "away_team": at,
            "home_seq": home_seq,
            "away_seq": away_seq,
            "home_pts_pre": float(r["home_pts_pre"]),
            "away_pts_pre": float(r["away_pts_pre"]),
            "home_rank_pre": float(r["home_rank_pre"]),
            "away_rank_pre": float(r["away_rank_pre"]),
            "home_rest_days": float(r["home_rest_days"]) if np.isfinite(r["home_rest_days"]) else np.nan,
            "away_rest_days": float(r["away_rest_days"]) if np.isfinite(r["away_rest_days"]) else np.nan,
            "target": r["target"],
        }

        if FORM_COLS:
            hform = safe_get_form(ht, i)
            aform = safe_get_form(at, i)
            for c in FORM_COLS:
                rec[f"home_{c}"] = float(hform[c]) if np.isfinite(hform[c]) else np.nan
                rec[f"away_{c}"] = float(aform[c]) if np.isfinite(aform[c]) else np.nan
                rec[f"diff_{c}"] = (rec[f"home_{c}"] - rec[f"away_{c}"]) if (
                    np.isfinite(rec[f"home_{c}"]) and np.isfinite(rec[f"away_{c}"])
                ) else np.nan

        out_rows.append(rec)

    return pd.DataFrame(out_rows)

data = build_dataset_rows().sort_values("match_idx").reset_index(drop=True)
print("Matches with enough history:", len(data), "out of", len(df0))

le = LabelEncoder()
y_enc = le.fit_transform(data["target"].values)
class_names = list(le.classes_)
print("Classes:", class_names)

# ============================================================
# 4) Time split
# ============================================================
split_idx = int(len(data) * 0.85)
train_df = data.iloc[:split_idx].copy()
test_df  = data.iloc[split_idx:].copy()

# ============================================================
# 5) Build CTX_COLS + impute/normalize using train stats only
# ============================================================
BASE_CTX = ["home_pts_pre","away_pts_pre","home_rank_pre","away_rank_pre","home_rest_days","away_rest_days"]

EXTRA_CTX = []
if FORM_COLS:
    EXTRA_CTX += [f"home_{c}" for c in FORM_COLS]
    EXTRA_CTX += [f"away_{c}" for c in FORM_COLS]
    EXTRA_CTX += [f"diff_{c}" for c in FORM_COLS]

CTX_COLS = BASE_CTX + EXTRA_CTX
print("CTX_DIM:", len(CTX_COLS))

train_meds = {}
for c in CTX_COLS:
    v = pd.to_numeric(train_df[c], errors="coerce")
    med = float(np.nanmedian(v.to_numpy()))
    train_meds[c] = med
    train_df[c] = v.fillna(med)
    test_df[c]  = pd.to_numeric(test_df[c], errors="coerce").fillna(med)

token_stack = np.concatenate(train_df["home_seq"].values.tolist() + train_df["away_seq"].values.tolist(), axis=0)
tok_mean = token_stack.mean(axis=0)
tok_std  = token_stack.std(axis=0) + 1e-6

ctx_mat = train_df[CTX_COLS].to_numpy(dtype=np.float32)
ctx_mean = ctx_mat.mean(axis=0)
ctx_std  = ctx_mat.std(axis=0) + 1e-6

def norm_tokens(x): return (x - tok_mean) / tok_std
def norm_ctx(x):    return (x - ctx_mean) / ctx_std

# ============================================================
# 6) Dataset / Dataloader
# ============================================================
class EPLSeqDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, y: np.ndarray):
        self.f = frame.reset_index(drop=True)
        self.y = y.astype(np.int64)

    def __len__(self):
        return len(self.f)

    def __getitem__(self, idx):
        r = self.f.iloc[idx]
        home_seq = norm_tokens(r["home_seq"]).astype(np.float32)
        away_seq = norm_tokens(r["away_seq"]).astype(np.float32)
        ctxv = norm_ctx(r[CTX_COLS].to_numpy(dtype=np.float32))
        home_id = team_to_id[r["home_team"]]
        away_id = team_to_id[r["away_team"]]
        return (
            torch.from_numpy(home_seq),
            torch.from_numpy(away_seq),
            torch.tensor(ctxv, dtype=torch.float32),
            torch.tensor(home_id, dtype=torch.long),
            torch.tensor(away_id, dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.long),
        )

y_train = le.transform(train_df["target"].values)
y_test  = le.transform(test_df["target"].values)

train_loader = DataLoader(EPLSeqDataset(train_df, y_train), batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
test_loader  = DataLoader(EPLSeqDataset(test_df,  y_test),  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# ============================================================
# 7) Model (Transformer + collision features)
# ============================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        T = x.size(1)
        return x + self.pe[:, :T, :]

class TrajTransformer(nn.Module):
    def __init__(self, token_dim, n_teams, ctx_dim, n_classes):
        super().__init__()
        self.team_emb = nn.Embedding(n_teams, D_MODEL)
        self.in_proj = nn.Linear(token_dim, D_MODEL)
        self.pos = PositionalEncoding(D_MODEL, max_len=SEQ_LEN)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=D_FF,
            dropout=DROPOUT, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=N_LAYERS)

        self.cls = nn.Parameter(torch.zeros(1, 1, D_MODEL))
        nn.init.normal_(self.cls, std=0.02)

        self.ctx_mlp = nn.Sequential(
            nn.Linear(ctx_dim, D_MODEL),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(D_MODEL, D_MODEL),
        )

        self.head = nn.Sequential(
            nn.Linear(D_MODEL*5, 256),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(256, n_classes)
        )

    def encode_team(self, seq, team_id):
        B, T, _ = seq.shape
        x = self.in_proj(seq)
        x = self.pos(x)
        cls = self.cls.expand(B, 1, -1)
        x = torch.cat([cls, x], dim=1)
        te = self.team_emb(team_id).unsqueeze(1)
        x = x + te
        z = self.encoder(x)
        return z[:, 0, :]

    def forward(self, home_seq, away_seq, ctx, home_id, away_id):
        h = self.encode_team(home_seq, home_id)
        a = self.encode_team(away_seq, away_id)
        c = self.ctx_mlp(ctx)
        diff = h - a
        prod = h * a
        feat = torch.cat([h, a, diff, prod, c], dim=1)
        return self.head(feat)

token_dim = len(TOKEN_COLS)
ctx_dim = len(CTX_COLS)
n_classes = len(class_names)

model = TrajTransformer(token_dim, n_teams=len(teams), ctx_dim=ctx_dim, n_classes=n_classes).to(DEVICE)

# ============================================================
# 8) Train (RESUMABLE) + Early stopping + checkpointing
# ============================================================
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING) if LABEL_SMOOTHING > 0 else nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    all_logits, all_y = [], []
    for home_seq, away_seq, ctxv, home_id, away_id, y in loader:
        home_seq = home_seq.to(DEVICE)
        away_seq = away_seq.to(DEVICE)
        ctxv = ctxv.to(DEVICE)
        home_id = home_id.to(DEVICE)
        away_id = away_id.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(home_seq, away_seq, ctxv, home_id, away_id)
        loss = criterion(logits, y)

        if train:
            optim.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        total_loss += float(loss.item()) * y.size(0)
        all_logits.append(logits.detach().cpu())
        all_y.append(y.detach().cpu())

    all_logits = torch.cat(all_logits, dim=0).numpy()
    all_y = torch.cat(all_y, dim=0).numpy()
    probs = torch.softmax(torch.from_numpy(all_logits), dim=1).numpy()
    return total_loss / len(loader.dataset), log_loss(all_y, probs)

def ckpt_path(epoch: int) -> str:
    return os.path.join(CKPT_DIR, f"{CKPT_PREFIX}_epoch_{epoch:03d}.pt")

def find_latest_checkpoint():
    files = glob.glob(os.path.join(CKPT_DIR, f"{CKPT_PREFIX}_epoch_*.pt"))
    if not files:
        return None
    def epnum(p):
        m = re.search(r"_epoch_(\d+)\.pt$", p)
        return int(m.group(1)) if m else -1
    files = sorted(files, key=epnum)
    return files[-1]

def _cleanup_old_checkpoints(keep_last_k: int):
    if keep_last_k <= 0:
        return
    files = glob.glob(os.path.join(CKPT_DIR, f"{CKPT_PREFIX}_epoch_*.pt"))
    if len(files) <= keep_last_k:
        return
    def epnum(p):
        m = re.search(r"_epoch_(\d+)\.pt$", p)
        return int(m.group(1)) if m else -1
    files = sorted(files, key=epnum)
    for p in files[:-keep_last_k]:
        try:
            os.remove(p)
        except OSError:
            pass

def save_checkpoint(epoch, best_test, best_state):
    payload = {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optim_state_dict": optim.state_dict(),
        "best_test": float(best_test),
        "best_state": best_state,
        "meta": {
            "RUN_NAME": RUN_NAME,
            "SEQ_LEN": SEQ_LEN,
            "TOKEN_COLS": TOKEN_COLS,
            "CTX_COLS": CTX_COLS,
            "D_MODEL": D_MODEL,
            "N_HEAD": N_HEAD,
            "N_LAYERS": N_LAYERS,
            "D_FF": D_FF,
            "DROPOUT": DROPOUT,
            "class_names": class_names,
            "team_to_id": team_to_id,
            "tok_mean": tok_mean,
            "tok_std": tok_std,
            "ctx_mean": ctx_mean,
            "ctx_std": ctx_std,
            "train_meds": train_meds,
        }
    }
    path = ckpt_path(epoch)
    torch.save(payload, path)
    print(f"💾 Saved checkpoint: {path}")
    _cleanup_old_checkpoints(SAVE_KEEP_LAST_K)

def save_best_snapshot(best_test, best_state):
    path = os.path.join(CKPT_DIR, f"{CKPT_PREFIX}_BEST.pt")
    torch.save({
        "model_state_dict": best_state,
        "best_test": float(best_test),
        "meta": {
            "RUN_NAME": RUN_NAME,
            "SEQ_LEN": SEQ_LEN,
            "TOKEN_COLS": TOKEN_COLS,
            "CTX_COLS": CTX_COLS,
            "class_names": class_names,
            "team_to_id": team_to_id,
            "tok_mean": tok_mean,
            "tok_std": tok_std,
            "ctx_mean": ctx_mean,
            "ctx_std": ctx_std,
            "train_meds": train_meds,
            "D_MODEL": D_MODEL,
            "N_HEAD": N_HEAD,
            "N_LAYERS": N_LAYERS,
            "D_FF": D_FF,
            "DROPOUT": DROPOUT,
        }
    }, path)
    print(f"🏁 Updated BEST snapshot: {path}")

def load_checkpoint(path: str):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optim.load_state_dict(ckpt["optim_state_dict"])

    start_epoch = int(ckpt["epoch"]) + 1
    best_test = float(ckpt.get("best_test", 1e9))
    best_state = ckpt.get("best_state", None)

    for state in optim.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(DEVICE)

    print(f"✅ Resumed from: {path}")
    print(f"   start_epoch={start_epoch}, best_test={best_test:.6f}")
    return start_epoch, best_test, best_state

start_epoch = 1
best_test = 1e9
best_state = None

if RESUME:
    latest = find_latest_checkpoint()
    if latest is not None:
        start_epoch, best_test, best_state = load_checkpoint(latest)
    else:
        print(f"ℹ️ No checkpoint found in {CKPT_DIR}. Training from scratch.")

if best_state is None:
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

no_improve = 0
end_epoch = start_epoch + EPOCHS - 1

for ep in range(start_epoch, end_epoch + 1):
    tr_loss, tr_ll = run_epoch(train_loader, train=True)
    te_loss, te_ll = run_epoch(test_loader, train=False)
    print(f"Epoch {ep:03d} | train loss {tr_loss:.4f} ll {tr_ll:.4f} | test loss {te_loss:.4f} ll {te_ll:.4f}")

    improved = te_ll < best_test - 1e-5
    if improved:
        best_test = te_ll
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
        print(f"⭐ New best test log loss: {best_test:.6f}")
    else:
        no_improve += 1

    if (ep % SAVE_EVERY_N == 0) or (ep == end_epoch):
        save_checkpoint(ep, best_test, best_state)

    if (ep % SAVE_BEST_EVERY_N == 0) or (ep == end_epoch):
        save_best_snapshot(best_test, best_state)

    if (ep >= MIN_EPOCHS_BEFORE_STOP) and (no_improve >= EARLY_STOP_PATIENCE):
        print(f"🛑 Early stopping: no improvement for {EARLY_STOP_PATIENCE} epochs.")
        break

model.load_state_dict(best_state)
print("✅ Best test log loss:", best_test)

# ============================================================
# 9) Temperature scaling calibration (on test split)
# ============================================================
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.logT = nn.Parameter(torch.zeros(()))

    def forward(self, logits):
        T = torch.exp(self.logT) + 1e-6
        return logits / T

scaler = TemperatureScaler().to(DEVICE)
optT = torch.optim.LBFGS(scaler.parameters(), lr=0.1, max_iter=50)

model.eval()
test_logits = []
test_y = []
with torch.no_grad():
    for home_seq, away_seq, ctxv, home_id, away_id, y in test_loader:
        logits = model(home_seq.to(DEVICE), away_seq.to(DEVICE), ctxv.to(DEVICE), home_id.to(DEVICE), away_id.to(DEVICE))
        test_logits.append(logits)
        test_y.append(y.to(DEVICE))

test_logits = torch.cat(test_logits, dim=0)
test_y = torch.cat(test_y, dim=0)

def closure():
    optT.zero_grad()
    loss = nn.CrossEntropyLoss()(scaler(test_logits), test_y)
    loss.backward()
    return loss

optT.step(closure)
T = float(torch.exp(scaler.logT).detach().cpu())
print("Calibrated Temperature T:", T)

FINAL_PATH = os.path.join(CKPT_DIR, f"{CKPT_PREFIX}_BEST_CALIBRATED.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "temperature_state_dict": scaler.state_dict(),
    "best_test": float(best_test),
    "meta": {
        "RUN_NAME": RUN_NAME,
        "SEQ_LEN": SEQ_LEN,
        "TOKEN_COLS": TOKEN_COLS,
        "CTX_COLS": CTX_COLS,
        "class_names": class_names,
        "team_to_id": team_to_id,
        "tok_mean": tok_mean,
        "tok_std": tok_std,
        "ctx_mean": ctx_mean,
        "ctx_std": ctx_std,
        "train_meds": train_meds,
        "D_MODEL": D_MODEL,
        "N_HEAD": N_HEAD,
        "N_LAYERS": N_LAYERS,
        "D_FF": D_FF,
        "DROPOUT": DROPOUT,
        "collision_features": ["h", "a", "h-a", "h*a", "ctx"],
        "available_stats": list(available_stats.keys()),
    }
}, FINAL_PATH)
print(f"💾 Saved BEST+CALIBRATED to: {FINAL_PATH}")

# ============================================================
# 10) TRUE AS-OF DATE fixture prediction (no leakage)
#   - as-of league table context
#   - as-of rolling form (last match for each team strictly before dt)
# ============================================================

# Build quick per-team match list (match_idx, date) for as-of queries
team_match_history = {}
for t in teams:
    hist = long_by_team[t][["match_idx","date"]].copy()
    hist["date"] = pd.to_datetime(hist["date"])
    team_match_history[t] = hist.sort_values(["date","match_idx"]).reset_index(drop=True)

def latest_team_seq_asof(team: str, asof_dt: pd.Timestamp):
    team = clean_team_name(team)
    hist = long_by_team.get(team, None)
    if hist is None:
        return None
    hist = hist.copy()
    hist["date"] = pd.to_datetime(hist["date"])
    past = hist[hist["date"] < pd.to_datetime(asof_dt)].sort_values(["date","match_idx"])
    if len(past) < SEQ_LEN:
        return None
    return past.iloc[-SEQ_LEN:][TOKEN_COLS].to_numpy(dtype=np.float32)

def build_table_state_asof(df0_in: pd.DataFrame, asof_dt: pd.Timestamp):
    dfx = df0_in.copy()
    dfx["date"] = pd.to_datetime(dfx["date"], errors="coerce")
    dfx = dfx[dfx["date"].notna()].copy()
    dfx["home_team"] = dfx["home_team"].map(clean_team_name)
    dfx["away_team"] = dfx["away_team"].map(clean_team_name)
    dfx = dfx[dfx["date"] < pd.to_datetime(asof_dt)].sort_values(["date"]).reset_index(drop=True)

    teams_local = sorted(pd.unique(pd.concat([df0_in["home_team"], df0_in["away_team"]], ignore_index=True)).tolist())
    teams_local = [clean_team_name(t) for t in teams_local]
    teams_local = sorted(list(dict.fromkeys(teams_local)))

    points_loc = {t: 0.0 for t in teams_local}
    gf_loc = {t: 0.0 for t in teams_local}
    ga_loc = {t: 0.0 for t in teams_local}
    last_date_loc = {t: None for t in teams_local}

    for _, r in dfx.iterrows():
        dt = r["date"]
        ht = clean_team_name(r["home_team"]); at = clean_team_name(r["away_team"])
        hg = pd.to_numeric(r["home_goals"], errors="coerce")
        ag = pd.to_numeric(r["away_goals"], errors="coerce")
        if not (np.isfinite(hg) and np.isfinite(ag)):
            continue
        hp, ap = result_points(float(hg), float(ag))
        points_loc[ht] += float(hp); points_loc[at] += float(ap)
        gf_loc[ht] += float(hg); ga_loc[ht] += float(ag)
        gf_loc[at] += float(ag); ga_loc[at] += float(hg)
        last_date_loc[ht] = dt; last_date_loc[at] = dt

    table = []
    for t in teams_local:
        gd = gf_loc[t] - ga_loc[t]
        table.append((t, points_loc[t], gd, gf_loc[t]))
    table_sorted = sorted(table, key=lambda x: (x[1], x[2], x[3]), reverse=True)
    rank_loc = {t: (rk + 1) for rk, (t, _, _, _) in enumerate(table_sorted)}

    return points_loc, gf_loc, ga_loc, last_date_loc, rank_loc

def last_match_idx_before(team: str, asof_dt: pd.Timestamp):
    team = clean_team_name(team)
    hist = team_match_history.get(team, None)
    if hist is None or len(hist) == 0:
        return None
    asof_dt = pd.to_datetime(asof_dt)
    past = hist[hist["date"] < asof_dt]
    if len(past) == 0:
        return None
    return int(past.iloc[-1]["match_idx"])

def fixture_ctx_asof(dt, ht, at):
    dt = pd.to_datetime(dt)
    ht = clean_team_name(ht); at = clean_team_name(at)

    points_loc, gf_loc, ga_loc, last_date_loc, rank_loc = build_table_state_asof(df0, dt)

    def rest(team):
        ld = last_date_loc.get(team, None)
        if ld is None:
            return np.nan
        return float((dt - pd.to_datetime(ld)).days)

    base = {
        "home_pts_pre": float(points_loc.get(ht, 0.0)),
        "away_pts_pre": float(points_loc.get(at, 0.0)),
        "home_rank_pre": float(rank_loc.get(ht, len(teams))),
        "away_rank_pre": float(rank_loc.get(at, len(teams))),
        "home_rest_days": rest(ht),
        "away_rest_days": rest(at),
    }

    feat = {}
    if FORM_COLS:
        h_idx = last_match_idx_before(ht, dt)
        a_idx = last_match_idx_before(at, dt)
        hform = safe_get_form(ht, h_idx) if h_idx is not None else {c: np.nan for c in FORM_COLS}
        aform = safe_get_form(at, a_idx) if a_idx is not None else {c: np.nan for c in FORM_COLS}
        for c in FORM_COLS:
            feat[f"home_{c}"] = hform.get(c, np.nan)
            feat[f"away_{c}"] = aform.get(c, np.nan)
            hv = feat[f"home_{c}"]; av = feat[f"away_{c}"]
            feat[f"diff_{c}"] = (hv - av) if (np.isfinite(hv) and np.isfinite(av)) else np.nan

    out = {}
    out.update(base)
    out.update(feat)

    vec = np.array([out.get(c, np.nan) for c in CTX_COLS], dtype=np.float32)
    for j, c in enumerate(CTX_COLS):
        if not np.isfinite(vec[j]):
            vec[j] = train_meds[c]
    return vec

def predict_fixture(dt, ht, at):
    dt = pd.to_datetime(dt)
    hs = latest_team_seq_asof(ht, dt)
    asq = latest_team_seq_asof(at, dt)
    if hs is None or asq is None:
        return None

    c = fixture_ctx_asof(dt, ht, at)
    hs_t = torch.from_numpy(norm_tokens(hs)).unsqueeze(0).to(DEVICE)
    as_t = torch.from_numpy(norm_tokens(asq)).unsqueeze(0).to(DEVICE)
    c_t  = torch.from_numpy(norm_ctx(c)).unsqueeze(0).to(DEVICE)

    hid = torch.tensor([team_to_id[clean_team_name(ht)]], dtype=torch.long).to(DEVICE)
    aid = torch.tensor([team_to_id[clean_team_name(at)]], dtype=torch.long).to(DEVICE)

    model.eval(); scaler.eval()
    with torch.no_grad():
        logits = model(hs_t, as_t, c_t, hid, aid)
        logits = scaler(logits)
        p = torch.softmax(logits, dim=1).cpu().numpy()[0]

    return dict(zip(class_names, p.tolist()))

rows_out = []
for _, r in fixtures_df.iterrows():
    dt, ht, at = r["date"], r["home_team"], r["away_team"]
    p = predict_fixture(dt, ht, at)
    if p is None:
        rows_out.append([dt, ht, at, np.nan, np.nan, np.nan, "NO DATA", np.nan])
        continue
    pH = p.get("H", np.nan); pD = p.get("D", np.nan); pA = p.get("A", np.nan)
    pick = max([("home",pH),("draw",pD),("away",pA)], key=lambda x: x[1])[0]
    conf = max(pH, pD, pA)
    rows_out.append([dt, ht, at, pH, pD, pA, pick, conf])

summary = pd.DataFrame(rows_out, columns=["date","home_team","away_team","p_home","p_draw","p_away","pick","confidence"])
print("\n================= NEXT WEEK SUMMARY (Transformer) =================")
print(summary.sort_values("confidence", ascending=False).reset_index(drop=True))

print("\n✅ Done.")

/tmp/ipykernel_10896/812441746.py:104: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df0["date"] = pd.to_datetime(df0["date"], errors="coerce")


Loaded: epl_all_seasons_football_data.csv
Rows after cleanup: 8006
Date range: 2005-01-01 -> 2026-12-02
Available stats for rolling form: ['shots', 'shots_on_target', 'corners', 'yellows', 'reds']
Matches with enough history: 7736 out of 8006
Classes: ['A', 'D', 'H']
CTX_DIM: 171
ℹ️ No checkpoint found in checkpoints_epl_traj/epl_traj_v3. Training from scratch.
Epoch 001 | train loss 1.0263 ll 1.0176 | test loss 1.0194 ll 1.0070
⭐ New best test log loss: 1.006986
Epoch 002 | train loss 0.9723 ll 0.9580 | test loss 0.9929 ll 0.9800
⭐ New best test log loss: 0.980012
Epoch 003 | train loss 0.9452 ll 0.9282 | test loss 0.9640 ll 0.9498
⭐ New best test log loss: 0.949805
Epoch 004 | train loss 0.9181 ll 0.8988 | test loss 0.9438 ll 0.9265
⭐ New best test log loss: 0.926512
Epoch 005 | train loss 0.8987 ll 0.8760 | test loss 0.9288 ll 0.9074
⭐ New best test log loss: 0.907384
Epoch 006 | train loss 0.8771 ll 0.8510 | test loss 0.9314 ll 0.9110
Epoch 007 | train loss 0.8707 ll 0.8431 | test 

In [3]:
import numpy as np
import pandas as pd
import torch

# --- sanity ---
assert "CTX_COLS" in globals(), "CTX_COLS missing. Run training cell first."
assert "ctx_mean" in globals() and "ctx_std" in globals(), "ctx_mean/ctx_std missing. Run training cell first."

# Ensure numpy arrays
CTX_COLS = list(CTX_COLS)
ctx_mean_np = np.asarray(ctx_mean, dtype=np.float32).copy()
ctx_std_np  = np.asarray(ctx_std,  dtype=np.float32).copy()

def norm_ctx_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return (x - ctx_mean_np) / ctx_std_np

def norm_tokens_np(x: np.ndarray) -> np.ndarray:
    return (x - tok_mean) / tok_std

def compute_rank_from_tables():
    teams_local = list(team_to_id.keys())
    table = []
    for t in teams_local:
        gd = float(gf_tot.get(t, 0.0) - ga_tot.get(t, 0.0))
        table.append((t, float(points.get(t, 0.0)), gd, float(gf_tot.get(t, 0.0))))
    table_sorted = sorted(table, key=lambda x: (x[1], x[2], x[3]), reverse=True)
    return {t: (rk + 1) for rk, (t, _, _, _) in enumerate(table_sorted)}

_final_rank_cache = compute_rank_from_tables()

def fixture_ctx_full(dt, ht, at):
    """
    Returns a RAW ctx vector with length == len(CTX_COLS).
    Strategy:
      - Start from ctx_mean (so unknown features become "typical")
      - Fill any recognized features (home/away pts, rank, rest days) if those names exist in CTX_COLS
    """
    dt = pd.to_datetime(dt)
    ht = clean_team_name(ht)
    at = clean_team_name(at)

    # Start from mean -> safe default for all 171 features
    c = ctx_mean_np.copy()

    # Build values we can compute causally
    def rest_days(team):
        ld = last_date.get(team, None)
        if ld is None or pd.isna(ld):
            return np.nan
        return float((dt - pd.to_datetime(ld)).days)

    known = {
        "home_pts_pre": float(points.get(ht, 0.0)),
        "away_pts_pre": float(points.get(at, 0.0)),
        "home_rank_pre": float(_final_rank_cache.get(ht, len(team_to_id))),
        "away_rank_pre": float(_final_rank_cache.get(at, len(team_to_id))),
        "home_rest_days": rest_days(ht),
        "away_rest_days": rest_days(at),
    }

    # Fill those into correct positions if present
    col_to_idx = {col: i for i, col in enumerate(CTX_COLS)}
    for k, v in known.items():
        if k in col_to_idx and np.isfinite(v):
            c[col_to_idx[k]] = float(v)

    # If rest days missing, keep it at mean (already in c). Optionally clip:
    for k in ["home_rest_days", "away_rest_days"]:
        if k in col_to_idx:
            idx = col_to_idx[k]
            c[idx] = float(np.clip(c[idx], 0.0, 30.0))

    return c.astype(np.float32)

def predict_fixture_transformer(dt, ht, at):
    hs = latest_team_seq(ht)
    asq = latest_team_seq(at)
    if hs is None or asq is None:
        return None

    c = fixture_ctx_full(dt, ht, at)

    hs_t = torch.from_numpy(norm_tokens_np(hs)).unsqueeze(0).to(DEVICE)
    as_t = torch.from_numpy(norm_tokens_np(asq)).unsqueeze(0).to(DEVICE)
    c_t  = torch.from_numpy(norm_ctx_np(c)).unsqueeze(0).to(DEVICE)

    hid = torch.tensor([team_to_id[clean_team_name(ht)]], dtype=torch.long).to(DEVICE)
    aid = torch.tensor([team_to_id[clean_team_name(at)]], dtype=torch.long).to(DEVICE)

    model.eval(); scaler.eval()
    with torch.no_grad():
        logits = model(hs_t, as_t, c_t, hid, aid)
        logits = scaler(logits)
        p = torch.softmax(logits, dim=1).cpu().numpy()[0]

    return dict(zip(class_names, p.tolist()))

print(f"✅ Patched: fixture_ctx_full returns ctx shape {len(CTX_COLS)} (ctx_mean={ctx_mean_np.shape}, ctx_std={ctx_std_np.shape})")

✅ Patched: fixture_ctx_full returns ctx shape 171 (ctx_mean=(171,), ctx_std=(171,))


In [4]:
# ============================================================
# Interactive "Next Game" UI — TRANSFORMER (Trajectory, NO-ODDS) — UPDATED (REALISTIC)
#
# ✅ Recomputes table context "AS-OF" the selected date (no leakage)
# ✅ Uses TrajTransformer + temperature scaler (calibrated)
# ✅ ctx vector matches training dimension len(CTX_COLS) (e.g., 171)  ✅ FIXED
# ✅ Better confidence: margin (top1-top2) + entropy
# ✅ EV/edge ONLY computed if you provide odds (REALISTIC DEFAULT)
# ✅ Filters: allow_draw, min_conf_margin, min_EV, min_edge, max_overround, min_odds/max_odds
# ✅ Optional: fractional Kelly sizing (only when odds provided)
#
# REQUIREMENTS (must exist from your training cell):
#   df0, clean_team_name
#   long_by_team, TOKEN_COLS, SEQ_LEN
#   team_to_id, class_names
#   CTX_COLS, tok_mean, tok_std, ctx_mean, ctx_std
#   model, scaler, DEVICE
# ============================================================

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import torch

# -------------------- sanity checks --------------------
needed = [
    "df0","clean_team_name","long_by_team","TOKEN_COLS","SEQ_LEN",
    "team_to_id","class_names","CTX_COLS","tok_mean","tok_std","ctx_mean","ctx_std",
    "model","scaler","DEVICE"
]
_missing = [x for x in needed if x not in globals()]
if _missing:
    raise RuntimeError(f"Missing required variables from training cell: {_missing}")

# Ensure numpy arrays
CTX_COLS = list(CTX_COLS)
ctx_mean_np = np.asarray(ctx_mean, dtype=np.float32).copy()
ctx_std_np  = np.asarray(ctx_std,  dtype=np.float32).copy()

if ctx_mean_np.shape[0] != len(CTX_COLS) or ctx_std_np.shape[0] != len(CTX_COLS):
    raise ValueError(f"CTX mismatch: len(CTX_COLS)={len(CTX_COLS)} ctx_mean={ctx_mean_np.shape} ctx_std={ctx_std_np.shape}")

# -------------------- small utils --------------------
def _is_finite(x):
    return x is not None and np.isfinite(x)

def _entropy(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1.0)
    return float(-(p * np.log(p)).sum())

def _confidence_margin(p_dict, labs=("H","D","A")):
    vals = np.array([float(p_dict.get(k, np.nan)) for k in labs], dtype=float)
    vals = np.where(np.isfinite(vals), vals, 0.0)
    s = np.sort(vals)[::-1]
    return float(s[0] - s[1]) if len(s) >= 2 else 0.0

# -------------------- odds parsing --------------------
def _parse_user_odds_to_decimal(s: str):
    s = (s or "").strip()
    if s == "":
        return None
    s2 = s.replace(" ", "")
    try:
        if s2.startswith("+") or s2.startswith("-"):
            a = float(s2)
            return 1.0 + (a / 100.0) if a > 0 else 1.0 + (100.0 / abs(a))

        x = float(s2)
        if 1.01 <= x <= 25.0:
            return float(x)  # decimal
        a = float(x)  # american
        return 1.0 + (a / 100.0) if a > 0 else 1.0 + (100.0 / abs(a))
    except Exception:
        return None

def implied_probs_from_decimal_odds(dec_h, dec_d, dec_a):
    if not (_is_finite(dec_h) and _is_finite(dec_d) and _is_finite(dec_a)):
        return None, None, None, None
    p_raw = np.array([1/dec_h, 1/dec_d, 1/dec_a], dtype=float)
    overround = float(p_raw.sum() - 1.0)
    p_fair = p_raw / (p_raw.sum() + 1e-12)
    return float(p_fair[0]), float(p_fair[1]), float(p_fair[2]), float(overround)

# -------------------- normalization helpers --------------------
def norm_tokens_np(x: np.ndarray) -> np.ndarray:
    return (x - tok_mean) / tok_std

def norm_ctx_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return (x - ctx_mean_np) / ctx_std_np

# -------------------- NO-LEAKAGE AS-OF table builder --------------------
def _result_points(hg, ag):
    if hg > ag:  return 3.0, 0.0
    if hg < ag:  return 0.0, 3.0
    return 1.0, 1.0

def build_table_state_asof(df0_in: pd.DataFrame, asof_dt: pd.Timestamp):
    dfx = df0_in.copy()
    dfx["date"] = pd.to_datetime(dfx["date"], errors="coerce")
    dfx = dfx[dfx["date"].notna()].copy()
    dfx["home_team"] = dfx["home_team"].map(clean_team_name)
    dfx["away_team"] = dfx["away_team"].map(clean_team_name)

    dfx = dfx[dfx["date"] < pd.to_datetime(asof_dt)].sort_values("date").reset_index(drop=True)

    teams_local = sorted(pd.unique(pd.concat([df0_in["home_team"], df0_in["away_team"]], ignore_index=True)).tolist())
    teams_local = [clean_team_name(t) for t in teams_local]
    teams_local = sorted(list(dict.fromkeys(teams_local)))

    points = {t: 0.0 for t in teams_local}
    gf_tot = {t: 0.0 for t in teams_local}
    ga_tot = {t: 0.0 for t in teams_local}
    last_date = {t: None for t in teams_local}

    have_goals = ("home_goals" in dfx.columns) and ("away_goals" in dfx.columns)

    for _, r in dfx.iterrows():
        dt = r["date"]
        ht = clean_team_name(r["home_team"])
        at = clean_team_name(r["away_team"])

        if have_goals:
            hg = pd.to_numeric(r["home_goals"], errors="coerce")
            ag = pd.to_numeric(r["away_goals"], errors="coerce")
            if not (np.isfinite(hg) and np.isfinite(ag)):
                continue
            hp, ap = _result_points(float(hg), float(ag))
            points[ht] += hp; points[at] += ap
            gf_tot[ht] += float(hg); ga_tot[ht] += float(ag)
            gf_tot[at] += float(ag); ga_tot[at] += float(hg)
        else:
            lab = None
            for c in ["ft_result","FTR","target"]:
                if c in dfx.columns:
                    lab = str(r[c]).strip().upper()
                    break
            if lab not in ["H","D","A"]:
                continue
            if lab == "H":
                points[ht] += 3.0
            elif lab == "A":
                points[at] += 3.0
            else:
                points[ht] += 1.0
                points[at] += 1.0

        last_date[ht] = dt
        last_date[at] = dt

    table = []
    for t in teams_local:
        gd = gf_tot[t] - ga_tot[t]
        table.append((t, points[t], gd, gf_tot[t]))
    table_sorted = sorted(table, key=lambda x: (x[1], x[2], x[3]), reverse=True)
    rank = {t: (rk + 1) for rk, (t, _, _, _) in enumerate(table_sorted)}

    return points, gf_tot, ga_tot, last_date, rank

def fixture_ctx_asof_full(asof_dt, ht, at, df0_in):
    """
    IMPORTANT: returns ctx length == len(CTX_COLS) (e.g. 171).
    We start from ctx_mean (safe default), then overwrite known fields
    IF those fields exist in CTX_COLS.
    """
    asof_dt = pd.to_datetime(asof_dt)
    ht = clean_team_name(ht); at = clean_team_name(at)

    points, gf_tot, ga_tot, last_date, rank = build_table_state_asof(df0_in, asof_dt)

    def rest(team):
        ld = last_date.get(team, None)
        if ld is None:
            return np.nan
        return float((asof_dt - pd.to_datetime(ld)).days)

    known = {
        "home_pts_pre": float(points.get(ht, 0.0)),
        "away_pts_pre": float(points.get(at, 0.0)),
        "home_rank_pre": float(rank.get(ht, len(team_to_id))),
        "away_rank_pre": float(rank.get(at, len(team_to_id))),
        "home_rest_days": rest(ht),
        "away_rest_days": rest(at),
    }

    c = ctx_mean_np.copy()
    col_to_idx = {col: i for i, col in enumerate(CTX_COLS)}
    for k, v in known.items():
        if k in col_to_idx and np.isfinite(v):
            c[col_to_idx[k]] = float(v)

    # rest-day impute/clip if present
    for k in ["home_rest_days", "away_rest_days"]:
        if k in col_to_idx:
            idx = col_to_idx[k]
            if not np.isfinite(c[idx]):
                c[idx] = ctx_mean_np[idx]
            c[idx] = float(np.clip(c[idx], 0.0, 30.0))

    return c.astype(np.float32)

# -------------------- transformer prediction --------------------
def latest_team_seq(team: str):
    team = clean_team_name(team)
    hist = long_by_team.get(team, None)
    if hist is None or len(hist) < SEQ_LEN:
        return None
    return hist.iloc[-SEQ_LEN:][TOKEN_COLS].to_numpy(dtype=np.float32)

def predict_fixture_transformer(dt, ht, at):
    hs = latest_team_seq(ht)
    asq = latest_team_seq(at)
    if hs is None or asq is None:
        return None

    c = fixture_ctx_asof_full(dt, ht, at, df0)  # ✅ FULL ctx (171)

    hs_t = torch.from_numpy(norm_tokens_np(hs)).unsqueeze(0).to(DEVICE)
    as_t = torch.from_numpy(norm_tokens_np(asq)).unsqueeze(0).to(DEVICE)
    c_t  = torch.from_numpy(norm_ctx_np(c)).unsqueeze(0).to(DEVICE)

    hid = torch.tensor([team_to_id[clean_team_name(ht)]], dtype=torch.long).to(DEVICE)
    aid = torch.tensor([team_to_id[clean_team_name(at)]], dtype=torch.long).to(DEVICE)

    model.eval(); scaler.eval()
    with torch.no_grad():
        logits = model(hs_t, as_t, c_t, hid, aid)
        logits = scaler(logits)
        p = torch.softmax(logits, dim=1).cpu().numpy()[0]

    return dict(zip(class_names, p.tolist()))

# -------------------- UI widgets --------------------
teams_ui = sorted(pd.unique(pd.concat([df0["home_team"], df0["away_team"]], ignore_index=True)).tolist())
teams_ui = [clean_team_name(t) for t in teams_ui]
teams_ui = sorted(list(dict.fromkeys(teams_ui)))

date_picker = widgets.DatePicker(description="Date:", value=pd.Timestamp.today().date())

home_dd = widgets.Dropdown(
    options=teams_ui, description="Home:",
    value=("Wolves" if "Wolves" in teams_ui else teams_ui[0]),
    layout=widgets.Layout(width="320px")
)
away_dd = widgets.Dropdown(
    options=teams_ui, description="Away:",
    value=("Aston Villa" if "Aston Villa" in teams_ui else teams_ui[min(1, len(teams_ui)-1)]),
    layout=widgets.Layout(width="320px")
)

odds_home = widgets.Text(description="Odds H:", value="", placeholder="e.g. +110 or 2.10",
                         layout=widgets.Layout(width="260px"))
odds_draw = widgets.Text(description="Odds D:", value="", placeholder="e.g. +240 or 3.40",
                         layout=widgets.Layout(width="260px"))
odds_away = widgets.Text(description="Odds A:", value="", placeholder="e.g. -120 or 1.83",
                         layout=widgets.Layout(width="260px"))

allow_draw = widgets.Checkbox(description="Allow Draw bets", value=True)

min_conf_margin = widgets.FloatSlider(
    description="Min conf (p1-p2)",
    min=0.0, max=0.40, step=0.01, value=0.05,
    readout_format=".2f", layout=widgets.Layout(width="420px")
)
min_edge = widgets.FloatSlider(
    description="Min edge",
    min=0.0, max=0.15, step=0.005, value=0.02,
    readout_format=".3f", layout=widgets.Layout(width="420px")
)
min_ev = widgets.FloatSlider(
    description="Min EV",
    min=-0.10, max=0.20, step=0.005, value=0.01,
    readout_format=".3f", layout=widgets.Layout(width="420px")
)
max_overround = widgets.FloatSlider(
    description="Max vig (overround)",
    min=0.00, max=0.20, step=0.005, value=0.08,
    readout_format=".3f", layout=widgets.Layout(width="420px")
)
min_odds = widgets.FloatSlider(
    description="Min odds",
    min=1.01, max=3.00, step=0.01, value=1.30,
    readout_format=".2f", layout=widgets.Layout(width="420px")
)
max_odds = widgets.FloatSlider(
    description="Max odds",
    min=2.00, max=15.00, step=0.10, value=6.0,
    readout_format=".1f", layout=widgets.Layout(width="420px")
)

use_kelly = widgets.Checkbox(description="Use fractional Kelly sizing", value=True)
kelly_frac = widgets.FloatSlider(
    description="Kelly fraction",
    min=0.0, max=1.0, step=0.05, value=0.25,
    readout_format=".2f", layout=widgets.Layout(width="420px")
)
bankroll = widgets.FloatText(description="Bankroll ($):", value=100.0,
                             layout=widgets.Layout(width="260px"))
flat_stake = widgets.FloatText(description="Flat stake ($):", value=2.0,
                               layout=widgets.Layout(width="260px"))

btn = widgets.Button(description="Predict", button_style="primary", icon="check")
out = widgets.Output()

def _kelly_fraction(p, dec):
    b = float(dec) - 1.0
    if b <= 0:
        return 0.0
    f = (float(dec) * float(p) - 1.0) / b
    return float(np.clip(f, 0.0, 1.0))

def on_click(_):
    with out:
        clear_output(wait=True)

        dt = date_picker.value
        ht = home_dd.value
        at = away_dd.value

        if dt is None:
            print("⚠️ Pick a date.")
            return
        if ht == at:
            print("⚠️ Home and Away can’t be the same team.")
            return

        p = predict_fixture_transformer(dt, ht, at)
        if p is None:
            print("⚠️ Not enough team history for SEQ_LEN =", SEQ_LEN, "for one/both teams.")
            return

        probs = pd.Series({k: float(p.get(k, 0.0)) for k in ["H","D","A"]})
        probs = probs / (probs.sum() + 1e-12)

        ent = _entropy(probs.values)
        margin = _confidence_margin(probs.to_dict(), labs=("H","D","A"))

        print(f"{pd.to_datetime(dt).date()} | {clean_team_name(ht)} vs {clean_team_name(at)}")
        print(f"Model probs: H={probs['H']:.3f}  D={probs['D']:.3f}  A={probs['A']:.3f}")
        print(f"Uncertainty: margin(p1-p2)={margin:.3f}  |  entropy={ent:.3f}")

        base_table = pd.DataFrame({
            "Outcome": ["Home (H)", "Draw (D)", "Away (A)"],
            "p_model": [float(probs["H"]), float(probs["D"]), float(probs["A"])],
        })
        # display(base_table.style.format({"p_model":"{:.3f}"}))

        if margin < float(min_conf_margin.value):
            print(f"Suggestion: NO BET ❌ (low confidence margin < {min_conf_margin.value:.3f})")
            return

        dec_h = _parse_user_odds_to_decimal(odds_home.value)
        dec_d = _parse_user_odds_to_decimal(odds_draw.value)
        dec_a = _parse_user_odds_to_decimal(odds_away.value)

        have_all_odds = _is_finite(dec_h) and _is_finite(dec_d) and _is_finite(dec_a)
        if not have_all_odds:
            print("Odds missing/invalid → EV/edge not computed (REALISTIC MODE).")
            print("Tip: type decimal (e.g., 2.10) or American (e.g., +110).")
            return

        pbh, pbd, pba, ov = implied_probs_from_decimal_odds(dec_h, dec_d, dec_a)
        print(f"Odds (decimal): H={dec_h:.3f}  D={dec_d:.3f}  A={dec_a:.3f}")
        print(f"Overround (vig): {ov:.3f}")

        if not np.isfinite(ov) or ov > float(max_overround.value):
            print(f"Suggestion: NO BET ❌ (vig too high > {max_overround.value:.3f})")
            return

        rows = []
        for lab, name, dec, pb in [
            ("H","Home (H)",dec_h,pbh),
            ("D","Draw (D)",dec_d,pbd),
            ("A","Away (A)",dec_a,pba),
        ]:
            if lab == "D" and not bool(allow_draw.value):
                continue
            p_model = float(probs[lab])
            edge = p_model - float(pb)
            ev = (p_model * float(dec)) - 1.0
            rows.append([lab, name, p_model, float(pb), edge, ev, float(dec)])

        table = pd.DataFrame(rows, columns=["lab","Outcome","p_model","p_book","edge","EV","dec_odds"])
        if len(table) == 0:
            print("No bettable outcomes.")
            return

        table = table[(table["dec_odds"] >= float(min_odds.value)) & (table["dec_odds"] <= float(max_odds.value))].copy()
        if len(table) == 0:
            print(f"Suggestion: NO BET ❌ (outside odds range {min_odds.value:.2f}–{max_odds.value:.2f})")
            return

        best = table.loc[table["EV"].idxmax()].copy()

        if float(best["EV"]) < float(min_ev.value) or float(best["edge"]) < float(min_edge.value):
            print(f"Best EV side: {best['Outcome']} | EV={best['EV']:.3f} | edge={best['edge']:.3f}")
            print("Suggestion: NO BET ❌ (fails EV/edge thresholds)")
            display(table.style.format({"p_model":"{:.3f}","p_book":"{:.3f}","edge":"{:.3f}","EV":"{:.3f}","dec_odds":"{:.3f}"}))
            return

        BR = float(bankroll.value) if bankroll.value is not None else 0.0
        flat = float(flat_stake.value) if flat_stake.value is not None else 0.0

        if use_kelly.value and BR > 0:
            f = _kelly_fraction(best["p_model"], best["dec_odds"])
            stake_amt = BR * float(kelly_frac.value) * f
            stake_note = f"Kelly f*={f:.3f}, fraction={kelly_frac.value:.2f} → stake=${stake_amt:.2f}"
        else:
            stake_amt = flat
            stake_note = f"Flat stake=${stake_amt:.2f}"

        print(f"Best EV side: {best['Outcome']} | EV={best['EV']:.3f} | edge={best['edge']:.3f}")
        print(f"Suggestion: BET ✅  |  {stake_note}")

        display(table.style.format({"p_model":"{:.3f}","p_book":"{:.3f}","edge":"{:.3f}","EV":"{:.3f}","dec_odds":"{:.3f}"}))

btn.on_click(on_click)

ui_row1 = widgets.HBox([date_picker, home_dd, away_dd])
ui_row2 = widgets.HBox([odds_home, odds_draw, odds_away])
ui_row3 = widgets.HBox([allow_draw, bankroll, flat_stake])
ui_row4 = widgets.HBox([min_conf_margin, min_edge])
ui_row5 = widgets.HBox([min_ev, max_overround])
ui_row6 = widgets.HBox([min_odds, max_odds])
ui_row7 = widgets.HBox([use_kelly, kelly_frac, btn])

display(ui_row1, ui_row2, ui_row3, ui_row4, ui_row5, ui_row6, ui_row7, out)

print("✅ Updated Transformer UI ready (ctx dimension fixed).")
print(f"CTX dim = {len(CTX_COLS)}")
print("Realistic mode: EV/edge computed ONLY if you enter odds.")

Output()

✅ Updated Transformer UI ready (ctx dimension fixed).
CTX dim = 171
Realistic mode: EV/edge computed ONLY if you enter odds.


In [6]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from ipywidgets import (
    FloatSlider, FloatText, IntSlider, Dropdown, Checkbox,
    VBox, HBox, Output, interactive_output
)

# -------------------- odds columns --------------------
ODDS_CANDIDATES = [
    ("odds_home","odds_draw","odds_away"),
    ("B365H","B365D","B365A"),
    ("PSH","PSD","PSA"),
    ("WHH","WHD","WHA"),
    ("VCH","VCD","VCA"),
]

def find_odds_triplet(df):
    for h,d,a in ODDS_CANDIDATES:
        if {h,d,a}.issubset(df.columns):
            return (h,d,a)
    return None

odds_cols = find_odds_triplet(df0)
if odds_cols is None:
    raise ValueError("No odds columns found in df0. Need odds_home/odds_draw/odds_away (or B365H/B365D/B365A etc).")

HODD, DODD, AODD = odds_cols
print("✅ Using odds columns:", odds_cols)

# -------------------- helpers --------------------
def implied_probs_from_decimal_odds(dec_h, dec_d, dec_a):
    if not (np.isfinite(dec_h) and np.isfinite(dec_d) and np.isfinite(dec_a)):
        return None
    p_raw = np.array([1/dec_h, 1/dec_d, 1/dec_a], dtype=float)
    overround = float(p_raw.sum() - 1.0)
    p_fair = p_raw / p_raw.sum()
    return float(p_fair[0]), float(p_fair[1]), float(p_fair[2]), overround

def model_probs_for_batch(home_seq, away_seq, ctx, home_id, away_id):
    model.eval(); scaler.eval()
    with torch.no_grad():
        logits = model(home_seq, away_seq, ctx, home_id, away_id)
        logits = scaler(logits)
        return torch.softmax(logits, dim=1)

label_to_i = {lab:i for i, lab in enumerate(class_names)}
for lab in ["H","D","A"]:
    if lab not in label_to_i:
        raise ValueError(f"class_names must include H/D/A. Got {class_names}")

def _choose_side_max_ev(row):
    evs = {"H": row["EV_H"], "D": row["EV_D"], "A": row["EV_A"]}
    side = max(evs.items(), key=lambda kv: kv[1])[0]
    return side

def _odds_for_side(row, side):
    return float(row[HODD] if side=="H" else (row[DODD] if side=="D" else row[AODD]))

def _p_for_side(row, side):
    return float(row["pH"] if side=="H" else (row["pD"] if side=="D" else row["pA"]))

def _edge_for_side(row, side):
    return float(row["edge_H"] if side=="H" else (row["edge_D"] if side=="D" else row["edge_A"]))

def _ev_for_side(row, side):
    return float(row["EV_H"] if side=="H" else (row["EV_D"] if side=="D" else row["EV_A"]))

def _kelly_fraction(p, dec_odds):
    # Kelly for decimal odds:
    # b = dec_odds - 1
    # f* = (p*(b+1) - 1)/b = (p*dec_odds - 1)/(dec_odds - 1)
    b = dec_odds - 1.0
    if b <= 0:
        return 0.0
    f = (p * dec_odds - 1.0) / b
    return float(max(0.0, f))

# -------------------- build sim table once (predictions + odds) --------------------
def build_scored_table(df0, data):
    cols_needed = ["date", "ft_result", HODD, DODD, AODD]
    miss = [c for c in cols_needed if c not in df0.columns]
    if miss:
        raise ValueError(f"df0 missing required columns: {miss}")

    meta = df0.loc[:, cols_needed].copy()
    meta["date"] = pd.to_datetime(meta["date"], errors="coerce")
    meta["ft_result"] = meta["ft_result"].astype(str).str.strip().str.upper()
    meta[[HODD, DODD, AODD]] = meta[[HODD, DODD, AODD]].apply(pd.to_numeric, errors="coerce")

    sim = data[["match_idx","home_team","away_team","home_seq","away_seq"] + CTX_COLS + ["target"]].copy()
    sim = sim.merge(
        meta.reset_index(drop=True).reset_index().rename(columns={"index":"match_idx"}),
        on="match_idx",
        how="left"
    )

    # basic filters
    sim = sim.dropna(subset=["date", HODD, DODD, AODD])
    sim = sim[sim["ft_result"].isin(["H","D","A"])].copy()

    # ids
    sim["home_id"] = sim["home_team"].map(lambda t: team_to_id[clean_team_name(t)])
    sim["away_id"] = sim["away_team"].map(lambda t: team_to_id[clean_team_name(t)])

    # tensors
    home_seq_arr = np.stack([norm_tokens(x).astype(np.float32) for x in sim["home_seq"].values], axis=0)
    away_seq_arr = np.stack([norm_tokens(x).astype(np.float32) for x in sim["away_seq"].values], axis=0)
    ctx_arr = np.stack([norm_ctx(sim.loc[i, CTX_COLS].to_numpy(dtype=np.float32)).astype(np.float32)
                        for i in sim.index], axis=0)

    home_seq_t = torch.from_numpy(home_seq_arr).to(DEVICE)
    away_seq_t = torch.from_numpy(away_seq_arr).to(DEVICE)
    ctx_t      = torch.from_numpy(ctx_arr).to(DEVICE)
    home_id_t  = torch.from_numpy(sim["home_id"].to_numpy(np.int64)).to(DEVICE)
    away_id_t  = torch.from_numpy(sim["away_id"].to_numpy(np.int64)).to(DEVICE)

    probs = model_probs_for_batch(home_seq_t, away_seq_t, ctx_t, home_id_t, away_id_t).detach().cpu().numpy()
    sim["pH"] = probs[:, label_to_i["H"]]
    sim["pD"] = probs[:, label_to_i["D"]]
    sim["pA"] = probs[:, label_to_i["A"]]

    # book fair probs + overround
    bookH, bookD, bookA, over = [], [], [], []
    for dh, dd, da in sim[[HODD, DODD, AODD]].to_numpy():
        out = implied_probs_from_decimal_odds(float(dh), float(dd), float(da))
        if out is None:
            bookH.append(np.nan); bookD.append(np.nan); bookA.append(np.nan); over.append(np.nan)
        else:
            bh, bd, ba, ov = out
            bookH.append(bh); bookD.append(bd); bookA.append(ba); over.append(ov)

    sim["p_book_H"] = bookH
    sim["p_book_D"] = bookD
    sim["p_book_A"] = bookA
    sim["overround"] = over

    # EV + edge
    sim["EV_H"] = sim["pH"] * sim[HODD] - 1.0
    sim["EV_D"] = sim["pD"] * sim[DODD] - 1.0
    sim["EV_A"] = sim["pA"] * sim[AODD] - 1.0

    sim["edge_H"] = sim["pH"] - sim["p_book_H"]
    sim["edge_D"] = sim["pD"] - sim["p_book_D"]
    sim["edge_A"] = sim["pA"] - sim["p_book_A"]

    sim = sim.sort_values("date").reset_index(drop=True)
    return sim

SIM_ALL = build_scored_table(df0, data)
print("✅ Scored rows:", len(SIM_ALL))

# -------------------- realistic betting sim --------------------
def run_strategy(
    sim_all: pd.DataFrame,
    start_date="2024-08-01",
    end_date=None,
    bankroll0=200.0,
    stake_method="Fractional Kelly",
    flat_stake=5.0,
    kelly_fraction=0.25,         # 0.25 = conservative
    max_stake_pct=0.03,          # cap per bet
    conf_thresh=0.45,            # model confidence filter (max prob)
    ev_thresh=0.01,              # require at least +1% EV
    edge_thresh=0.02,            # require edge >= 2% vs book fair
    max_odds=4.5,                # avoid longshots
    min_odds=1.30,               # avoid tiny payouts
    max_overround=0.08,          # avoid high vig markets
    allow_draw=True,
    use_odds_cap=True
):
    if end_date is None:
        end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

    sim = sim_all.copy()
    sim = sim[(sim["date"] >= pd.to_datetime(start_date)) & (sim["date"] <= pd.to_datetime(end_date))].copy()

    # restrict outcomes allowed (many bettors skip draws)
    if not allow_draw:
        # we’ll still compute EVs, but we will only consider H/A as candidates
        pass

    bankroll = float(bankroll0)
    bankroll_path = []
    bet_amounts = []
    bet_sides = []
    do_bets = []
    profits = []

    for _, r in sim.iterrows():
        # candidate sides
        if allow_draw:
            side = _choose_side_max_ev(r)
        else:
            # pick best EV among H/A only
            evs = {"H": r["EV_H"], "A": r["EV_A"]}
            side = max(evs.items(), key=lambda kv: kv[1])[0]

        p = _p_for_side(r, side)
        dec = _odds_for_side(r, side)
        edge = _edge_for_side(r, side)
        ev = _ev_for_side(r, side)
        conf = float(max(r["pH"], r["pD"], r["pA"]))  # confidence = max prob

        # realism filters
        ok = True
        if not np.isfinite(r["overround"]) or r["overround"] > float(max_overround):
            ok = False
        if ev < float(ev_thresh):
            ok = False
        if edge < float(edge_thresh):
            ok = False
        if conf < float(conf_thresh):
            ok = False

        if use_odds_cap:
            if dec > float(max_odds) or dec < float(min_odds):
                ok = False

        if not ok or bankroll <= 0:
            do_bets.append(False)
            bet_amounts.append(0.0)
            bet_sides.append(side)
            profits.append(0.0)
            bankroll_path.append(bankroll)
            continue

        # stake sizing
        if stake_method == "Flat":
            stake = float(flat_stake)
        else:
            f = _kelly_fraction(p, dec)
            stake = bankroll * float(kelly_fraction) * f

        # cap stake as % bankroll
        cap = bankroll * float(max_stake_pct)
        stake = float(np.clip(stake, 0.0, cap))

        # if stake becomes tiny, skip
        if stake < 0.50:
            do_bets.append(False)
            bet_amounts.append(0.0)
            bet_sides.append(side)
            profits.append(0.0)
            bankroll_path.append(bankroll)
            continue

        # settle bet
        win = (r["ft_result"] == side)
        profit = stake * (dec - 1.0) if win else -stake
        bankroll += profit

        do_bets.append(True)
        bet_amounts.append(stake)
        bet_sides.append(side)
        profits.append(profit)
        bankroll_path.append(bankroll)

    sim = sim.reset_index(drop=True)
    sim["bet_side"] = bet_sides
    sim["do_bet"] = do_bets
    sim["stake"] = bet_amounts
    sim["profit"] = profits
    sim["bankroll"] = bankroll_path
    sim["cum_profit"] = sim["profit"].cumsum()

    n_bets = int(sim["do_bet"].sum())
    total_staked = float(sim.loc[sim["do_bet"], "stake"].sum())
    net_profit = float(sim["cum_profit"].iloc[-1]) if len(sim) else 0.0
    roi = (net_profit / total_staked) if total_staked > 0 else 0.0

    summary = {
        "start_date": str(pd.to_datetime(start_date).date()),
        "end_date": str(pd.to_datetime(end_date).date()),
        "bets": n_bets,
        "total_staked": total_staked,
        "net_profit": net_profit,
        "ROI_on_staked": roi,
        "final_bankroll": float(sim["bankroll"].iloc[-1]) if len(sim) else bankroll0,
        "avg_stake": float(sim.loc[sim["do_bet"], "stake"].mean()) if n_bets else 0.0,
        "odds_cols": odds_cols,
    }
    return summary, sim

# -------------------- plotting --------------------
def plot_results(sim, title):
    plt.rcParams.update({
        "font.family": "serif",
        "axes.linewidth": 2.0,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

    fig, ax = plt.subplots(figsize=(12, 5), dpi=200)
    ax.plot(sim["date"], sim["bankroll"], linewidth=2.5)
    ax.axhline(sim["bankroll"].iloc[0], linestyle="--", linewidth=1.2)

    ax.set_title(title, pad=12)
    ax.set_xlabel("Date")
    ax.set_ylabel("Bankroll ($)")

    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

# -------------------- interactive UI --------------------
out = Output()

def dashboard(
    bankroll0,
    stake_method,
    flat_stake,
    kelly_fraction,
    max_stake_pct,
    conf_thresh,
    ev_thresh,
    edge_thresh,
    min_odds,
    max_odds,
    max_overround,
    allow_draw,
    use_odds_cap
):
    with out:
        out.clear_output(wait=True)
        summ, sim = run_strategy(
            SIM_ALL,
            start_date="2025-08-01",
            end_date=None,
            bankroll0=bankroll0,
            stake_method=stake_method,
            flat_stake=flat_stake,
            kelly_fraction=kelly_fraction,
            max_stake_pct=max_stake_pct,
            conf_thresh=conf_thresh,
            ev_thresh=ev_thresh,
            edge_thresh=edge_thresh,
            min_odds=min_odds,
            max_odds=max_odds,
            max_overround=max_overround,
            allow_draw=allow_draw,
            use_odds_cap=use_odds_cap
        )
        print(pd.Series(summ).to_string())
        title = (
            f"Realistic Bankroll Trajectory | {stake_method} | "
            f"conf≥{conf_thresh:.2f}, EV≥{ev_thresh:.2%}, edge≥{edge_thresh:.2%}"
        )
        plot_results(sim, title)

controls = {
    "bankroll0": FloatText(value=200.0, description="Start $"),
    "stake_method": Dropdown(options=["Fractional Kelly", "Flat"], value="Fractional Kelly", description="Stake"),
    "flat_stake": FloatSlider(value=5.0, min=1.0, max=50.0, step=1.0, description="Flat $"),
    "kelly_fraction": FloatSlider(value=0.25, min=0.05, max=0.50, step=0.05, description="Kelly frac"),
    "max_stake_pct": FloatSlider(value=0.03, min=0.01, max=0.10, step=0.01, description="Max %/bet"),
    "conf_thresh": FloatSlider(value=0.45, min=0.33, max=0.65, step=0.01, description="Conf min"),
    "ev_thresh": FloatSlider(value=0.01, min=0.00, max=0.05, step=0.005, description="EV min"),
    "edge_thresh": FloatSlider(value=0.02, min=0.00, max=0.06, step=0.005, description="Edge min"),
    "min_odds": FloatSlider(value=1.30, min=1.01, max=2.50, step=0.05, description="Min odds"),
    "max_odds": FloatSlider(value=4.50, min=2.00, max=15.0, step=0.5, description="Max odds"),
    "max_overround": FloatSlider(value=0.08, min=0.00, max=0.15, step=0.01, description="Max vig"),
    "allow_draw": Checkbox(value=True, description="Allow draws"),
    "use_odds_cap": Checkbox(value=True, description="Use odds caps"),
}

ui = VBox([
    HBox([controls["bankroll0"], controls["stake_method"]]),
    HBox([controls["flat_stake"], controls["kelly_fraction"], controls["max_stake_pct"]]),
    HBox([controls["conf_thresh"], controls["ev_thresh"], controls["edge_thresh"]]),
    HBox([controls["min_odds"], controls["max_odds"], controls["max_overround"]]),
    HBox([controls["allow_draw"], controls["use_odds_cap"]]),
])

display(ui, out)

interactive_output(dashboard, controls)

✅ Using odds columns: ('B365H', 'B365D', 'B365A')
✅ Scored rows: 7736


Output()

Output()